<a href="https://colab.research.google.com/github/ekachaip-COYG/sandbox/blob/main/ResizeFiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import shutil
import time
from PIL import Image
from PyPDF2 import PdfReader, PdfWriter
from datetime import datetime

# --- ชื่อไฟล์คอนฟิก ---
CONFIG_FILE = "config.txt"

def get_config():
    """โหลดค่าคอนฟิกจากไฟล์ ถ้าไม่มีให้สร้างใหม่"""
    default_config = {
        "SOURCE_DIR": "C:/Users/YourName/Documents/Work", # โฟลเดอร์ที่รวมไฟล์ PDF/รูป
        "BACKUP_DIR": "D:/Backups/FileResizer",           # โฟลเดอร์สำรองข้อมูล
        "RETENTION_DAYS": "7",                            # จำนวนวันที่เก็บ Backup
        "MIN_AGE_DAYS": "30"                              # อายุน้อยที่สุดของไฟล์ที่จะโดน Resize
    }

    if not os.path.exists(CONFIG_FILE):
        with open(CONFIG_FILE, "w", encoding="utf-8") as f:
            for key, val in default_config.items():
                f.write(f"{key}={val}\n")
        print(f"⚙️ [Config] สร้างไฟล์ {CONFIG_FILE} แล้ว กรุณาแก้ไข Path ในไฟล์นั้นก่อนใช้งาน")
        return default_config

    config = {}
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if "=" in line:
                key, val = line.strip().split("=", 1)
                config[key] = val
    return config

def clear_old_backups(backup_dir, days):
    """ลบ Backup เก่า"""
    if not os.path.exists(backup_dir): return
    now = time.time()
    retention_seconds = int(days) * 24 * 60 * 60
    for filename in os.listdir(backup_dir):
        file_path = os.path.join(backup_dir, filename)
        if os.path.isfile(file_path) and os.path.getmtime(file_path) < (now - retention_seconds):
            os.remove(file_path)
            print(f"🧹 ลบ Backup เก่า: {filename}")

def process_file(file_path, config):
    # ตรวจสอบอายุก่อน (Modified Date)
    file_time = os.path.getmtime(file_path)
    min_age_seconds = int(config["MIN_AGE_DAYS"]) * 24 * 60 * 60

    if file_time > (time.time() - min_age_seconds):
        return # ข้ามถ้าไฟล์ยังใหม่

    ext = os.path.splitext(file_path)[1].lower()
    if ext not in ['.jpg', '.jpeg', '.png', '.pdf']: return

    # ทำ Backup
    if not os.path.exists(config["BACKUP_DIR"]): os.makedirs(config["BACKUP_DIR"])
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(config["BACKUP_DIR"], f"{os.path.basename(file_path)}_{timestamp}.bak")
    shutil.copy2(file_path, backup_path)

    # Resize (ทับไฟล์เดิมหรือสร้างไฟล์ใหม่ตามต้องการ)
    try:
        output_path = file_path # ในที่นี้เลือกทับไฟล์เดิมเพราะเรามี Backup แล้ว
        if ext == '.pdf':
            reader = PdfReader(file_path)
            writer = PdfWriter()
            for page in reader.pages:
                page.compress_content_streams()
                writer.add_page(page)
            with open(output_path, "wb") as f:
                writer.write(f)
        else:
            img = Image.open(file_path)
            img.convert("RGB").save(output_path, "JPEG", quality=50, optimize=True)
        print(f"✅ [Done] บีบอัดไฟล์สำเร็จ: {os.path.basename(file_path)}")
    except Exception as e:
        print(f"❌ [Error] {file_path}: {e}")

def main():
    conf = get_config()
    source = conf["SOURCE_DIR"]

    print(f"🚀 เริ่มทำงานที่ Drive/Folder: {source}")
    clear_old_backups(conf["BACKUP_DIR"], conf["RETENTION_DAYS"])

    if not os.path.exists(source):
        print(f"🛑 ไม่พบโฟลเดอร์ต้นทาง: {source} (กรุณาแก้ใน config.txt)")
        return

    for root, dirs, files in os.walk(source):
        for file in files:
            process_file(os.path.join(root, file), conf)

    print("\n✨ จัดการไฟล์เก่ากว่า 1 เดือนเสร็จเรียบร้อย!")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'PyPDF2'

Now that Google Drive is mounted, let's create a `config.txt` file with paths that are compatible with Google Colab and your mounted drive. I'll use default folders like `Colab_FileResizer_Source` and `Colab_FileResizer_Backup` within your `My Drive` folder. You can modify these paths in the code below if you prefer different locations, or edit the `config.txt` file directly after it's created.

In [6]:
import os

CONFIG_FILE = "config.txt"

def write_config(source_dir, backup_dir):
    with open(CONFIG_FILE, "w", encoding="utf-8") as f:
        f.write(f"SOURCE_DIR={source_dir}\n")
        f.write(f"BACKUP_DIR={backup_dir}\n")
        f.write("RETENTION_DAYS=7\n")
        f.write("MIN_AGE_DAYS=30\n")
    print(f"⚙️ [Config] {CONFIG_FILE} updated with Google Drive paths.")

# Define Google Drive paths
google_drive_base = '/content/drive/My Drive/'
source_folder = os.path.join(google_drive_base, 'Colab_FileResizer_Source')
backup_folder = os.path.join(google_drive_base, 'Colab_FileResizer_Backup')

# Write the configuration file
write_config(source_folder, backup_folder)


⚙️ [Config] config.txt updated with Google Drive paths.


Next, let's create these new folders in your Google Drive if they don't already exist. This ensures the main script can find the source and backup directories.

In [7]:
import os

CONFIG_FILE = "config.txt"

def get_config_paths():
    config_paths = {}
    if os.path.exists(CONFIG_FILE):
        with open(CONFIG_FILE, "r", encoding="utf-8") as f:
            for line in f:
                if "=" in line:
                    key, val = line.strip().split("=", 1)
                    config_paths[key] = val
    return config_paths

config_data = get_config_paths()
source_dir = config_data.get('SOURCE_DIR')
backup_dir = config_data.get('BACKUP_DIR')

if source_dir and not os.path.exists(source_dir):
    os.makedirs(source_dir)
    print(f"Created SOURCE_DIR: {source_dir}")

if backup_dir and not os.path.exists(backup_dir):
    os.makedirs(backup_dir)
    print(f"Created BACKUP_DIR: {backup_dir}")

print("Google Drive source and backup directories are ready.")


Created SOURCE_DIR: /content/drive/My Drive/Colab_FileResizer_Source
Created BACKUP_DIR: /content/drive/My Drive/Colab_FileResizer_Backup
Google Drive source and backup directories are ready.


Now that `PyPDF2` is installed, `config.txt` is created with Google Drive paths, and the necessary directories exist, you can run the original main script cell (`FGnNwOKYByJ_`).

Please make sure to upload any PDF or image files you want to process into the `Colab_FileResizer_Source` folder (or your chosen `SOURCE_DIR`) in your Google Drive before running the script.

In [4]:
pip install PyPDF2

To load data from Google Drive, you first need to mount your Google Drive. This will allow your Colab notebook to access files stored in your Drive.

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


After running the above cell and following the authentication steps, your Google Drive will be mounted at `/content/drive`. You can then access your files using paths like `/content/drive/My Drive/YourFolder/your_file.csv`.

In [ ]:
import os
import shutil
import time
from PIL import Image
from PyPDF2 import PdfReader, PdfWriter
from datetime import datetime

# --- ชื่อไฟล์คอนฟิก ---
CONFIG_FILE = "config.txt"

def get_config():
    """โหลดค่าคอนฟิกจากไฟล์ ถ้าไม่มีให้สร้างใหม่"""
    default_config = {
        "SOURCE_DIR": "C:/Users/YourName/Documents/Work", # โฟลเดอร์ที่รวมไฟล์ PDF/รูป
        "BACKUP_DIR": "D:/Backups/FileResizer",           # โฟลเดอร์สำรองข้อมูล
        "RETENTION_DAYS": "7",                            # จำนวนวันที่เก็บ Backup
        "MIN_AGE_DAYS": "30"                              # อายุน้อยที่สุดของไฟล์ที่จะโดน Resize
    }

    if not os.path.exists(CONFIG_FILE):
        with open(CONFIG_FILE, "w", encoding="utf-8") as f:
            for key, val in default_config.items():
                f.write(f"{key}={val}\n")
        print(f"⚙️ [Config] สร้างไฟล์ {CONFIG_FILE} แล้ว กรุณาแก้ไข Path ในไฟล์นั้นก่อนใช้งาน")
        return default_config

    config = {}
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if "=" in line:
                key, val = line.strip().split("=", 1)
                config[key] = val
    return config

def clear_old_backups(backup_dir, days):
    """ลบ Backup เก่า"""
    if not os.path.exists(backup_dir): return
    now = time.time()
    retention_seconds = int(days) * 24 * 60 * 60
    for filename in os.listdir(backup_dir):
        file_path = os.path.join(backup_dir, filename)
        if os.path.isfile(file_path) and os.path.getmtime(file_path) < (now - retention_seconds):
            os.remove(file_path)
            print(f"🧹 ลบ Backup เก่า: {filename}")

def process_file(file_path, config):
    # ตรวจสอบอายุก่อน (Modified Date)
    file_time = os.path.getmtime(file_path)
    min_age_seconds = int(config["MIN_AGE_DAYS"]) * 24 * 60 * 60

    if file_time > (time.time() - min_age_seconds):
        return # ข้ามถ้าไฟล์ยังใหม่

    ext = os.path.splitext(file_path)[1].lower()
    if ext not in ['.jpg', '.jpeg', '.png', '.pdf']: return

    # ทำ Backup
    if not os.path.exists(config["BACKUP_DIR"]): os.makedirs(config["BACKUP_DIR"])
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(config["BACKUP_DIR"], f"{os.path.basename(file_path)}_{timestamp}.bak")
    shutil.copy2(file_path, backup_path)

    # Resize (ทับไฟล์เดิมหรือสร้างไฟล์ใหม่ตามต้องการ)
    try:
        output_path = file_path # ในที่นี้เลือกทับไฟล์เดิมเพราะเรามี Backup แล้ว
        if ext == '.pdf':
            reader = PdfReader(file_path)
            writer = PdfWriter()
            for page in reader.pages:
                page.compress_content_streams()
                writer.add_page(page)
            with open(output_path, "wb") as f:
                writer.write(f)
        else:
            img = Image.open(file_path)
            img.convert("RGB").save(output_path, "JPEG", quality=50, optimize=True)
        print(f"✅ [Done] บีบอัดไฟล์สำเร็จ: {os.path.basename(file_path)}")
    except Exception as e:
        print(f"❌ [Error] {file_path}: {e}")

def main():
    conf = get_config()
    source = conf["SOURCE_DIR"]

    print(f"🚀 เริ่มทำงานที่ Drive/Folder: {source}")
    clear_old_backups(conf["BACKUP_DIR"], conf["RETENTION_DAYS"])

    if not os.path.exists(source):
        print(f

In [ ]:
import os
import shutil
import time
from PIL import Image
from PyPDF2 import PdfReader, PdfWriter
from datetime import datetime

# --- ชื่อไฟล์คอนฟิก ---
CONFIG_FILE = "config.txt"

def get_config():
    """โหลดค่าคอนฟิกจากไฟล์ ถ้าไม่มีให้สร้างใหม่"""
    default_config = {
        "SOURCE_DIR": "C:/Users/YourName/Documents/Work", # โฟลเดอร์ที่รวมไฟล์ PDF/รูป
        "BACKUP_DIR": "D:/Backups/FileResizer",           # โฟลเดอร์สำรองข้อมูล
        "RETENTION_DAYS": "7",                            # จำนวนวันที่เก็บ Backup
        "MIN_AGE_DAYS": "30"                              # อายุน้อยที่สุดของไฟล์ที่จะโดน Resize
    }

    if not os.path.exists(CONFIG_FILE):
        with open(CONFIG_FILE, "w", encoding="utf-8") as f:
            for key, val in default_config.items():
                f.write(f"{key}={val}\n")
        print(f"⚙️ [Config] สร้างไฟล์ {CONFIG_FILE} แล้ว กรุณาแก้ไข Path ในไฟล์นั้นก่อนใช้งาน")
        return default_config

    config = {}
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if "=" in line:
                key, val = line.strip().split("=", 1)
                config[key] = val
    return config

def clear_old_backups(backup_dir, days):
    """ลบ Backup เก่า"""
    if not os.path.exists(backup_dir): return
    now = time.time()
    retention_seconds = int(days) * 24 * 60 * 60
    for filename in os.listdir(backup_dir):
        file_path = os.path.join(backup_dir, filename)
        if os.path.isfile(file_path) and os.path.getmtime(file_path) < (now - retention_seconds):
            os.remove(file_path)
            print(f"🧹 ลบ Backup เก่า: {filename}")

def process_file(file_path, config):
    # ตรวจสอบอายุก่อน (Modified Date)
    file_time = os.path.getmtime(file_path)
    min_age_seconds = int(config["MIN_AGE_DAYS"]) * 24 * 60 * 60

    if file_time > (time.time() - min_age_seconds):
        return # ข้ามถ้าไฟล์ยังใหม่

    ext = os.path.splitext(file_path)[1].lower()
    if ext not in ['.jpg', '.jpeg', '.png', '.pdf']: return

    # ทำ Backup
    if not os.path.exists(config["BACKUP_DIR"]): os.makedirs(config["BACKUP_DIR"])
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(config["BACKUP_DIR"], f"{os.path.basename(file_path)}_{timestamp}.bak")
    shutil.copy2(file_path, backup_path)

    # Resize (ทับไฟล์เดิมหรือสร้างไฟล์ใหม่ตามต้องการ)
    try:
        output_path = file_path # ในที่นี้เลือกทับไฟล์เดิมเพราะเรามี Backup แล้ว
        if ext == '.pdf':
            reader = PdfReader(file_path)
            writer = PdfWriter()
            for page in reader.pages:
                page.compress_content_streams()
                writer.add_page(page)
            with open(output_path, "wb") as f:
                writer.write(f)
        else:
            img = Image.open(file_path)
            img.convert("RGB").save(output_path, "JPEG", quality=50, optimize=True)
        print(f"✅ [Done] บีบอัดไฟล์สำเร็จ: {os.path.basename(file_path)}")
    except Exception as e:
        print(f"❌ [Error] {file_path}: {e}")

def main():
    conf = get_config()
    source = conf["SOURCE_DIR"]

    print(f"🚀 เริ่มทำงานที่ Drive/Folder: {source}")
    clear_old_backups(conf["BACKUP_DIR"], conf["RETENTION_DAYS"])

    if not os.path.exists(source):
        print(f"🛑 ไม่พบโฟลเดอร์ต้นทาง: {source} (กรุณาแก้ใน config.txt)")
        return

    for root, dirs, files in os.walk(source):
        for file in files:
            process_file(os.path.join(root, file), conf)

    print("\n✨ จัดการไฟล์เก่ากว่า 1 เดือนเสร็จเรียบร้อย!")

if __name__ == "__main__":
    main()

In [10]:
import os
import shutil
import time
from PIL import Image
from PyPDF2 import PdfReader, PdfWriter
from datetime import datetime

# --- ชื่อไฟล์คอนฟิก ---
CONFIG_FILE = "config.txt"

def get_config():
    """โหลดค่าคอนฟิกจากไฟล์ ถ้าไม่มีให้สร้างใหม่"""
    default_config = {
        "SOURCE_DIR": "https://drive.google.com/drive/u/0/folders/14oStvyo8C_FVfX3aA7YSwlZ6xnaw2-fz", # โฟลเดอร์ที่รวมไฟล์ PDF/รูป
        "BACKUP_DIR": "https://drive.google.com/drive/u/0/folders/1xEnRjlSE6hUYhXgszjVtXpdQ_2j87DUI",           # โฟลเดอร์สำรองข้อมูล
        "RETENTION_DAYS": "7",                            # จำนวนวันที่เก็บ Backup
        "MIN_AGE_DAYS": "1"                              # อายุน้อยที่สุดของไฟล์ที่จะโดน Resize
    }

    if not os.path.exists(CONFIG_FILE):
        with open(CONFIG_FILE, "w", encoding="utf-8") as f:
            for key, val in default_config.items():
                f.write(f"{key}={val}\n")
        print(f"⚙️ [Config] สร้างไฟล์ {CONFIG_FILE} แล้ว กรุณาแก้ไข Path ในไฟล์นั้นก่อนใช้งาน")
        return default_config

    config = {}
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if "=" in line:
                key, val = line.strip().split("=", 1)
                config[key] = val
    return config

def clear_old_backups(backup_dir, days):
    """ลบ Backup เก่า"""
    if not os.path.exists(backup_dir): return
    now = time.time()
    retention_seconds = int(days) * 24 * 60 * 60
    for filename in os.listdir(backup_dir):
        file_path = os.path.join(backup_dir, filename)
        if os.path.isfile(file_path) and os.path.getmtime(file_path) < (now - retention_seconds):
            os.remove(file_path)
            print(f"🧹 ลบ Backup เก่า: {filename}")

def process_file(file_path, config):
    # ตรวจสอบอายุก่อน (Modified Date)
    file_time = os.path.getmtime(file_path)
    min_age_seconds = int(config["MIN_AGE_DAYS"]) * 24 * 60 * 60

    if file_time > (time.time() - min_age_seconds):
        return # ข้ามถ้าไฟล์ยังใหม่

    ext = os.path.splitext(file_path)[1].lower()
    if ext not in ['.jpg', '.jpeg', '.png', '.pdf']: return

    # ทำ Backup
    if not os.path.exists(config["BACKUP_DIR"]): os.makedirs(config["BACKUP_DIR"])
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(config["BACKUP_DIR"], f"{os.path.basename(file_path)}_{timestamp}.bak")
    shutil.copy2(file_path, backup_path)

    # Resize (ทับไฟล์เดิมหรือสร้างไฟล์ใหม่ตามต้องการ)
    try:
        output_path = file_path # ในที่นี้เลือกทับไฟล์เดิมเพราะเรามี Backup แล้ว
        if ext == '.pdf':
            reader = PdfReader(file_path)
            writer = PdfWriter()
            for page in reader.pages:
                page.compress_content_streams()
                writer.add_page(page)
            with open(output_path, "wb") as f:
                writer.write(f)
        else:
            img = Image.open(file_path)
            img.convert("RGB").save(output_path, "JPEG", quality=50, optimize=True)
        print(f"✅ [Done] บีบอัดไฟล์สำเร็จ: {os.path.basename(file_path)}")
    except Exception as e:
        print(f"❌ [Error] {file_path}: {e}")

def main():
    conf = get_config()
    source = conf["SOURCE_DIR"]

    print(f"🚀 เริ่มทำงานที่ Drive/Folder: {source}")
    clear_old_backups(conf["BACKUP_DIR"], conf["RETENTION_DAYS"])

    if not os.path.exists(source):
        print(f"🛑 ไม่พบโฟลเดอร์ต้นทาง: {source} (กรุณาแก้ใน config.txt)")
        return

    for root, dirs, files in os.walk(source):
        for file in files:
            process_file(os.path.join(root, file), conf)

    print("\n✨ จัดการไฟล์เก่ากว่า 1 เดือนเสร็จเรียบร้อย!")

if __name__ == "__main__":
    main()


🚀 เริ่มทำงานที่ Drive/Folder: /content/drive/My Drive/Colab_FileResizer_Source

✨ จัดการไฟล์เก่ากว่า 1 เดือนเสร็จเรียบร้อย!
